In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import gc
import torch

# Delete any old model variables if they exist in the namespace
if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

# Force garbage collection and empty the PyTorch cache
gc.collect()
torch.cuda.empty_cache()

In [3]:
import os, math
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup,
)
import wandb  
import logging
logging.getLogger("wandb").setLevel(logging.ERROR)
os.environ["WANDB_SILENT"] = "true"


## Config


In [4]:
DATA_DIR        = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
MODEL_NAME      = "microsoft/deberta-v3-base"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN         = 256
BATCH_SIZE      = 4
ACCUM_STEPS     = 4         
EPOCHS          = 9
LR_BACKBONE     = 9e-6      
LR_HEAD         = 1e-4      
WARMUP_FRAC     = 0.1
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.05     
USE_FP16        = DEVICE == "cuda"
OPTION_COLS     = ["A", "B", "C", "D", "E"]

print(f"Device : {DEVICE}  |  FP16: {USE_FP16}")
print(f"Model  : {MODEL_NAME}")
print(f"Epochs : {EPOCHS}  |  Effective batch: {BATCH_SIZE * ACCUM_STEPS}")

Device : cuda  |  FP16: True
Model  : microsoft/deberta-v3-base
Epochs : 9  |  Effective batch: 16


In [5]:
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="24f2000817-t22026",
    name="deberta-v3-base",
    config=dict(
        model=MODEL_NAME, max_len=MAX_LEN,
        batch_size=BATCH_SIZE, accum_steps=ACCUM_STEPS,
        effective_batch=BATCH_SIZE * ACCUM_STEPS,
        epochs=EPOCHS, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
        warmup_frac=WARMUP_FRAC, grad_clip=GRAD_CLIP,
        label_smoothing=LABEL_SMOOTHING, fp16=USE_FP16,
    ),
)

In [6]:
def apk(actual, predicted, k=3):
    if not actual: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

## DATA

In [7]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

Train: 2000  |  Test: 500


In [8]:
# DATASET

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True):
        self.df        = df.reset_index(drop=True)
        self.tok       = tokenizer
        self.has_labels= has_labels
        self.lmap      = {c: i for i, c in enumerate(OPTION_COLS)}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        q       = str(row["prompt"])
        choices = [str(row[c]) for c in OPTION_COLS]

        enc = self.tok(
            [q] * 5, choices,
            truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        item = {k: v for k, v in enc.items()}
        if self.has_labels:
            item["labels"] = torch.tensor(self.lmap[str(row["answer"])], dtype=torch.long)
        return item

In [9]:
# MODEL

print("\nLoading model …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
model     = model.float()

train_ds = MCQDataset(train_df, tokenizer, has_labels=True)
test_ds  = MCQDataset(test_df,  tokenizer, has_labels=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


Loading model …


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                

## DIFFERENTIAL LR OPTIMIZER

In [10]:
head_params    = ["classifier", "pooler"]
no_decay       = ["bias", "LayerNorm.weight"]

optimizer_groups = [
    # backbone, with weight decay
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)
                and not any(hp in n for hp in head_params)],
     "lr": LR_BACKBONE, "weight_decay": 0.01},
    # backbone, no weight decay
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)
                and not any(hp in n for hp in head_params)],
     "lr": LR_BACKBONE, "weight_decay": 0.0},
    # head, with weight decay
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)
                and any(hp in n for hp in head_params)],
     "lr": LR_HEAD, "weight_decay": 0.01},
    # head, no weight decay
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)
                and any(hp in n for hp in head_params)],
     "lr": LR_HEAD, "weight_decay": 0.0},
    ]

optimizer   = AdamW(optimizer_groups, eps=1e-6)
total_steps = (len(train_dl) // ACCUM_STEPS) * EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

# Label smoothing loss
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)



## TRAINING

In [11]:
print("\n" + "="*50)
print("Fine-tuning …")
print("="*50)

best_loss = float("inf")
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_dl):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with torch.amp.autocast("cuda", enabled=USE_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # use label-smoothed loss instead of model's built-in CE
            loss = criterion(outputs.logits, labels) / ACCUM_STEPS

        if not math.isfinite(loss.item() * ACCUM_STEPS):
            optimizer.zero_grad()
            continue
        scaler.scale(loss).backward()

        # update every ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()  
            global_step += 1

            wandb.log({
                "train/step_loss": loss.item() * ACCUM_STEPS,
                "train/lr":        scheduler.get_last_lr()[0],
            }, step=global_step)

        total_loss += loss.item() * ACCUM_STEPS
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (step + 1) % 100 == 0:
            avg = total_loss / (step + 1)
            print(f"  Ep{epoch+1} step {step+1}/{len(train_dl)}  "
                  f"loss={avg:.4f}  acc={correct/total:.4f}  "
                  f"lr_backbone={scheduler.get_last_lr()[0]:.2e}")

    epoch_loss = total_loss / len(train_dl)
    epoch_acc  = correct / max(total, 1)
    print(f"\nEpoch {epoch+1}/{EPOCHS} — loss: {epoch_loss:.4f}  acc: {epoch_acc:.4f}\n")

    wandb.log({
        "train/epoch_loss": epoch_loss,
        "train/epoch_acc":  epoch_acc,
        "epoch": epoch + 1,
    }, step=global_step)

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")
        print(f"  ✓ Best model saved (loss={best_loss:.4f})")
        wandb.run.summary["best_epoch_loss"] = best_loss


Fine-tuning …


/tmp/ipykernel_24/930308463.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  Ep1 step 100/500  loss=1.6201  acc=0.1950  lr_backbone=2.01e-06
  Ep1 step 200/500  loss=1.6083  acc=0.2225  lr_backbone=4.02e-06
  Ep1 step 300/500  loss=1.6058  acc=0.2325  lr_backbone=6.03e-06
  Ep1 step 400/500  loss=1.5968  acc=0.2444  lr_backbone=8.04e-06
  Ep1 step 500/500  loss=1.5945  acc=0.2420  lr_backbone=9.00e-06

Epoch 1/9 — loss: 1.5945  acc: 0.2420

  ✓ Best model saved (loss=1.5945)
  Ep2 step 100/500  loss=1.4741  acc=0.3900  lr_backbone=8.97e-06
  Ep2 step 200/500  loss=1.4680  acc=0.3762  lr_backbone=8.91e-06
  Ep2 step 300/500  loss=1.4288  acc=0.4025  lr_backbone=8.83e-06
  Ep2 step 400/500  loss=1.3747  acc=0.4256  lr_backbone=8.73e-06
  Ep2 step 500/500  loss=1.3199  acc=0.4570  lr_backbone=8.59e-06

Epoch 2/9 — loss: 1.3199  acc: 0.4570

  ✓ Best model saved (loss=1.3199)
  Ep3 step 100/500  loss=1.0593  acc=0.5975  lr_backbone=8.44e-06
  Ep3 step 200/500  loss=1.0247  acc=0.6175  lr_backbone=8.26e-06
  Ep3 step 300/500  loss=1.0170  acc=0.6192  lr_backbone=8

## INFERENCE  (load best checkpoint)


In [12]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pt"))
print("\nLoaded best checkpoint for inference.")

def predict_top3(loader):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_FP16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs.logits.float().cpu().numpy())
    logits = np.vstack(all_logits)
    return logits, [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3] for r in logits]

print("Evaluating on training set …")
_, train_preds = predict_top3(DataLoader(train_ds, batch_size=BATCH_SIZE))
train_map3 = mapk(train_df["answer"].tolist(), train_preds)
print(f"Train MAP@3: {mapk(train_df['answer'].tolist(), train_preds):.4f}")
wandb.run.summary["train_map3"] = train_map3

print("\nGenerating test predictions …")
_, test_preds = predict_top3(test_dl)

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
submission.to_csv("/kaggle/working/submission_deberta_v3.csv", index=False)
print("✅  Saved → /kaggle/working/submission_deberta_v3.csv")
print(submission.head(10).to_string(index=False))
wandb.finish()


Loaded best checkpoint for inference.
Evaluating on training set …
Train MAP@3: 0.9881

Generating test predictions …
✅  Saved → /kaggle/working/submission_deberta_v3.csv
 ID Prediction
  1      A D C
  2      B D C
  3      B E A
  4      E C A
  5      C D A
  6      D B C
  7      E D C
  8      B E C
  9      C D E
 10      B E C
